# nanoGPT Kaggle 实战：对照官方源码一步步跑通

这份 notebook 对照 Andrej Karpathy 的 [nanoGPT](https://github.com/karpathy/nanoGPT) 官方仓库来跑。不要一上来训练大模型，先按官方 README 的 quick start 跑一个 Shakespeare 字符级 GPT。这样你能先看清楚完整链路：

```text
下载源码 -> 准备数据 -> 看源码结构 -> 写 Kaggle 小配置 -> 训练 -> 采样生成
```

建议 Kaggle 设置：

```text
右侧 Settings -> Accelerator -> GPU T4 或 P100
Internet -> On
```

如果你只是先看流程，也可以用 CPU，但训练会慢一些。这里的训练配置故意很小，目标是跑通和理解，不是追求最好效果。


## 0. 环境检查

先看 Kaggle 有没有 GPU。后面训练命令会自动选择 `cuda` 或 `cpu`。官方 nanoGPT 默认适合命令行项目，这里我们用 notebook cell 一步步调用它。


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


## 1. 下载 nanoGPT 官方源码

这一步对应官方 README：先拿到代码仓库。为了让 notebook 结果可复现，我们 checkout 到本文参考的提交：

```text
3adf61e154c3fe3fca428ad6bc3818b27a3b8291
```

如果 Kaggle Internet 关了，这一步会失败。那就需要把 nanoGPT 仓库作为 Kaggle Dataset 上传，再把路径改成本地 Dataset 路径。


In [ ]:
REPO_URL = "https://github.com/karpathy/nanoGPT.git"
NANOGPT_COMMIT = "3adf61e154c3fe3fca428ad6bc3818b27a3b8291"
WORKDIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
REPO_DIR = WORKDIR / "nanoGPT"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "fetch", "--depth", "1", "origin", NANOGPT_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", NANOGPT_COMMIT], cwd=REPO_DIR, check=True)

print("nanoGPT path:", REPO_DIR)
print("commit:")
subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True)


## 2. 先看项目结构

我们不急着训练，先看官方仓库里有哪些文件。nanoGPT 最重要的是：

```text
model.py   GPT 模型结构
train.py   训练主循环
sample.py  生成文本
data/      数据准备脚本
config/    训练配置
```


In [ ]:
for p in ["model.py", "train.py", "sample.py", "config/train_shakespeare_char.py", "data/shakespeare_char/prepare.py"]:
    path = REPO_DIR / p
    print(f"{p:40s}", "OK" if path.exists() else "MISSING")

print("\nTop-level files:")
for p in sorted(REPO_DIR.iterdir()):
    print(" ", p.name)


## 3. 安装最小依赖

`sample.py` 会导入 `tiktoken`。字符级 Shakespeare 采样其实会优先用 `meta.pkl`，但 `sample.py` 顶部还是会 import，所以这里提前装一下。


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=True)
print("tiktoken installed")


## 4. 对照官方 README：准备 Shakespeare 字符数据

官方 quick start 的第一步是：

```bash
python data/shakespeare_char/prepare.py
```

这段脚本会下载一个小的 Shakespeare 文本，然后做三件事：

```text
1. 找出文本里出现过的所有字符。
2. 给每个字符分配一个整数编号。
3. 保存 train.bin、val.bin 和 meta.pkl。
```

字符级模型的 token 就是单个字符；GPT-2 的 token 则是 BPE 子词片段。初学先跑字符级，更容易看懂。


In [ ]:
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "data/shakespeare_char/prepare.py"], check=True)


## 5. 看看 prepare.py 生成了什么

`train.bin` 和 `val.bin` 是一长串整数。`meta.pkl` 保存字符和编号的对应关系。

如果文本是：

```text
hello
```

训练时会构造：

```text
x = h e l l
y = e l l o
```

也就是输入和答案右移一位。语言模型学习的就是“下一个 token 是什么”。


In [ ]:
import pickle
import numpy as np

data_dir = REPO_DIR / "data" / "shakespeare_char"
train_bin = data_dir / "train.bin"
val_bin = data_dir / "val.bin"
meta_path = data_dir / "meta.pkl"

with open(meta_path, "rb") as f:
    meta = pickle.load(f)

print("vocab_size:", meta["vocab_size"])
print("train.bin bytes:", train_bin.stat().st_size)
print("val.bin bytes:", val_bin.stat().st_size)
print("first 20 chars in vocab:", "".join(meta["itos"][i] for i in range(min(20, meta["vocab_size"]))))

train_data = np.memmap(train_bin, dtype=np.uint16, mode="r")
print("first 80 token ids:", train_data[:80].tolist())
print("decoded preview:")
print("".join(meta["itos"][int(i)] for i in train_data[:200]))


## 6. 亲手模拟 get_batch

`train.py` 里的 `get_batch` 是训练数据入口。它随机从 `train.bin` 里切一段长度为 `block_size` 的 token 作为 `x`，再把它右移一位作为 `y`。

这一步非常重要，因为它说明了：训练 GPT 不需要人工标注“问题和答案”，普通文本自己就能构造出训练题。


In [ ]:
block_size_demo = 32
batch_size_demo = 2
ix = torch.randint(len(train_data) - block_size_demo, (batch_size_demo,))

x = torch.stack([torch.from_numpy((train_data[i:i+block_size_demo]).astype(np.int64)) for i in ix])
y = torch.stack([torch.from_numpy((train_data[i+1:i+1+block_size_demo]).astype(np.int64)) for i in ix])

print("x shape:", tuple(x.shape))
print("y shape:", tuple(y.shape))

itos = meta["itos"]
for row in range(batch_size_demo):
    print("\n--- sample", row, "---")
    print("x:", "".join(itos[int(i)] for i in x[row]))
    print("y:", "".join(itos[int(i)] for i in y[row]))


## 7. 对照 model.py：GPTConfig 是施工图

先看源码里的配置类。它不是模型本身，而是告诉模型要建多大：

```text
block_size：最多看多长上下文
vocab_size：词表大小
n_layer：多少个 Transformer block
n_head：每层多少个注意力头
n_embd：每个 token 向量多宽
dropout：训练时随机丢弃比例
```


In [ ]:
from pathlib import Path

model_py = (REPO_DIR / "model.py").read_text(encoding="utf-8")

def show_between(text, start_marker, end_marker):
    start = text.index(start_marker)
    end = text.index(end_marker, start)
    print(text[start:end].rstrip())

show_between(model_py, "@dataclass\nclass GPTConfig", "\nclass GPT(nn.Module):")


## 8. 对照 model.py：CausalSelfAttention 是核心

这一层对应 GPT-2 图解里的 masked self-attention。

注意这两行：

```python
self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
self.c_proj = nn.Linear(config.n_embd, config.n_embd)
```

`c_attn` 一次性生成 Q、K、V；`c_proj` 把多个注意力头的结果融合回模型维度。


In [ ]:
show_between(model_py, "class CausalSelfAttention", "\nclass MLP")


## 9. 重点解释：Q、K、V 的形状变化

源码里最容易卡住的是这几行：

```python
q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
```

假设小模型：

```text
B = 64
T = 256
C = 384
n_head = 6
head_size = 64
```

那么形状变化是：

```text
x:              B x T x C
c_attn(x):      B x T x 3C
split 后 q/k/v: B x T x C
view 后:        B x T x n_head x head_size
transpose 后:   B x n_head x T x head_size
```

这样每个 head 就能独立做注意力。


In [ ]:
B, T, C, n_head = 2, 8, 384, 6
head_size = C // n_head

fake_x = torch.randn(B, T, C)
fake_qkv = torch.randn(B, T, 3 * C)
q, k, v = fake_qkv.split(C, dim=2)
k2 = k.view(B, T, n_head, head_size).transpose(1, 2)

print("x:", tuple(fake_x.shape))
print("qkv:", tuple(fake_qkv.shape))
print("q:", tuple(q.shape))
print("k after view+transpose:", tuple(k2.shape))


## 10. 对照 model.py：MLP 和 Block

MLP 对应 GPT-2 图解里的前馈网络：

```text
n_embd -> 4*n_embd -> n_embd
```

Block 把 LayerNorm、Attention、MLP、残差连接串起来：

```python
x = x + self.attn(self.ln_1(x))
x = x + self.mlp(self.ln_2(x))
```

这就是“先交流，再加工”的一层 GPT block。


In [ ]:
show_between(model_py, "class MLP", "\n@dataclass")


## 11. 对照 model.py：GPT 类把所有零件组装起来

重点看这几块：

```text
wte：token embedding
wpe：position embedding
h：很多个 Block
ln_f：最后 LayerNorm
lm_head：输出词表 logits
```

这和 GPT-2 图解的主路径一致：

```text
token id -> embedding + position -> blocks -> lm_head -> logits
```


In [ ]:
show_between(model_py, "class GPT(nn.Module):", "\n    def get_num_params")


## 12. 写一个 Kaggle 小配置

官方 `config/train_shakespeare_char.py` 已经能跑，但我们在 Kaggle notebook 里先用更短训练步数，方便一步步观察。

这里不是追求最优效果，而是先让完整流程跑通。


In [ ]:
KAGGLE_CONFIG = REPO_DIR / "config" / "kaggle_shakespeare_char.py"

config_text = '''
out_dir = 'out-kaggle-shakespeare-char'
eval_interval = 100
eval_iters = 20
log_interval = 10
always_save_checkpoint = True

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 32
block_size = 128

n_layer = 4
n_head = 4
n_embd = 128
dropout = 0.1

learning_rate = 1e-3
max_iters = 600
lr_decay_iters = 600
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 50

compile = False
'''.strip() + "\n"

KAGGLE_CONFIG.write_text(config_text, encoding="utf-8")

print(KAGGLE_CONFIG.read_text(encoding="utf-8"))


## 13. 开始训练

这一步会调用官方 `train.py`。如果你开了 Kaggle GPU，通常会比较快。

先不要把 `max_iters` 调太大。等确认能跑通，再逐步加大：

```text
n_layer
n_head
n_embd
block_size
max_iters
```


In [ ]:
train_cmd = [
    sys.executable, "train.py", "config/kaggle_shakespeare_char.py",
    f"--device={DEVICE}",
    "--compile=False",
]
print("Running:", " ".join(train_cmd))
subprocess.run(train_cmd, cwd=REPO_DIR, check=True)


## 14. 训练后检查 checkpoint

训练会在 `out-kaggle-shakespeare-char/ckpt.pt` 保存模型。

checkpoint 不只是模型参数，还包括：

```text
model：模型权重
optimizer：优化器状态
model_args：模型结构参数
iter_num：训练步数
best_val_loss：最好验证损失
config：训练配置
```


In [ ]:
ckpt_path = REPO_DIR / "out-kaggle-shakespeare-char" / "ckpt.pt"
print("checkpoint exists:", ckpt_path.exists())
print("checkpoint size MB:", ckpt_path.stat().st_size / 1024 / 1024 if ckpt_path.exists() else None)

ckpt = torch.load(ckpt_path, map_location="cpu")
print("checkpoint keys:", ckpt.keys())
print("model args:", ckpt["model_args"])
print("iter_num:", ckpt["iter_num"])
print("best_val_loss:", ckpt["best_val_loss"])


## 15. 对照 sample.py：生成文本

官方生成脚本是：

```bash
python sample.py --out_dir=out-kaggle-shakespeare-char
```

它会加载 checkpoint，读取 `meta.pkl` 做字符级 decode，然后调用 `model.generate`。


In [ ]:
sample_cmd = [
    sys.executable, "sample.py",
    "--out_dir=out-kaggle-shakespeare-char",
    f"--device={DEVICE}",
    "--compile=False",
    "--num_samples=3",
    "--max_new_tokens=500",
    "--temperature=0.8",
    "--top_k=50",
]
print("Running:", " ".join(sample_cmd))
subprocess.run(sample_cmd, cwd=REPO_DIR, check=True)


## 16. generate 源码精读：为什么能一直写下去

`generate` 的核心循环是：

```text
取当前上下文
-> 前向计算 logits
-> 只拿最后一个位置
-> softmax 成概率
-> 抽样一个 token
-> 拼回输入
-> 继续下一轮
```

这就是自回归生成。模型不是一次写完整篇文章，而是一步一步预测下一个 token。


In [ ]:
start = model_py.index("    @torch.no_grad()\n    def generate")
print(model_py[start:].rstrip())


## 17. 下一步怎么举一反三

跑通以后，你可以改三类东西：

```text
1. 数据：换成自己的 input.txt。
2. 模型：调 n_layer、n_head、n_embd、block_size。
3. 训练：调 max_iters、batch_size、learning_rate、dropout。
```

初学建议顺序：

```text
先跑通默认小配置
再增加 max_iters
再增加 n_embd
再增加 n_layer
最后再考虑更复杂 tokenizer
```

如果你想训练中文，字符级也能跑，但效果有限。更好的路线是使用 tokenizer，把中文文本切成 token id，再复用 nanoGPT 的训练主循环。


## 18. 和前面 GPT-2 图解的对照

| GPT-2 图解概念 | nanoGPT 代码 |
|---|---|
| token embedding | `self.transformer.wte` |
| position embedding | `self.transformer.wpe` |
| 多层 Transformer block | `self.transformer.h` |
| masked self-attention | `CausalSelfAttention` |
| Q/K/V 一次性生成 | `self.c_attn` |
| 多头注意力 | `view(... n_head ...)` |
| 输出投影 | `self.c_proj` |
| MLP | `class MLP` |
| 残差连接 | `x = x + ...` |
| 输出词表 logits | `self.lm_head` |
| loss | `F.cross_entropy` |
| 生成 | `generate` |

这份 notebook 只是第一版：先把官方源码在 Kaggle 上跑通，再读懂每个关键代码块。真正训练大模型前，先把这条小链路跑顺，比盲目堆参数更重要。
